# 08i — Explain (Attention + GNNExplainer, DGCNN / SortPooling Readout)

Post-hoc explainability for `07i`'s trained checkpoints. **Requires
`07i` to have already been run** -- this notebook loads its saved
`{tag}_best_model.pt` / `{tag}_best_model_stats.pt` per scenario, it
does not train anything itself. Structurally identical to `08g`
(same two techniques, same point-selection logic, same normalization
handling) -- see `08g`'s intro for the shared mechanics. This intro
covers only what's specific to explaining a `readout="dgcnn"` model.

**The question this notebook exists to answer.** `docs/07g_07i_architecture.md`
§4 flags that `DGCNNReadout` gives **no anchor guarantee** -- unlike
`07g`'s `pool_anchor`, which always concatenates the incident/ego node's
own embedding into the graph vector, DGCNN's SortPooling ranks every
node by salience and keeps only the top-`k`; the anchor can rank outside
that cutoff and be dropped entirely. This was flagged as a real
possibility, not assumed either way. GNNExplainer's learned
node-importance mask gives a direct way to check: **does the anchor
node's importance collapse for `07i` relative to `07g`, and does that
correlate with wrong predictions (FP/FN)?**

**`k` reconstruction.** `DGCNNReadout`'s Conv1d/Linear layer shapes
depend on `k`, which `07i` computed dynamically from real data (the
40th-percentile rule, per encoder) rather than reading it from a fixed
config value -- so this notebook must recompute the SAME `k` values
before it can even construct the right architecture to load
`best_model.pt`'s state_dict into. The diagnostic cell below is copied
verbatim from `07i` (same sample, same `random_state=42`) -- as long as
the underlying dataset hasn't changed, this reproduces the exact `k`
`07i` actually trained with.

**Attention extraction is UNCHANGED by the readout swap** -- both
branches use `conv_type="gatv2"`, and attention lives in the
message-passing layers, not the readout. Any difference you see between
`08g` and `08i`'s attention-weight patterns reflects the readout
indirectly shaping what the encoder learns to attend to, not a direct
mechanical effect.

GPU recommended for the GNNExplainer optimization loop.

In [ ]:
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# TEMP: install locally patched src/ files until pushed to GitHub.
# Skip this cell once the repo itself is updated -- needs explain.py,
# train.py (best_model_stats.pt persistence), models.py
# (DGCNNReadout + readout="dgcnn"), plus graph_datasets.py,
# unified_graph.py, baseline_features.py, evaluate.py, plot_history.py.
from google.colab import files
import shutil

print("Upload explain.py, train.py, models.py, graph_datasets.py, unified_graph.py, "
      "baseline_features.py, evaluate.py, plot_history.py:")
uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, f"{REPO_DIR}/src/{fname}")
print("Patched files installed:", list(uploaded.keys()))

In [ ]:
!pip install -q torch_geometric xgboost scikit-learn scipy pyyaml pandas tqdm

In [ ]:
import yaml
from pathlib import Path
import torch

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)
with open(f"{REPO_DIR}/configs/model_dgcnn_comparison.yaml") as f:
    model_cfg = yaml.safe_load(f)

CITIES = paths_cfg["cities"]
INTERIM_DIR = Path(paths_cfg["interim_dir"])
COMBINED_PROCESSED_DIR = Path(paths_cfg["processed_dir"])
OUTPUTS_DIR = Path(paths_cfg["outputs_dir"])
# MUST match 07i's own dir names exactly.
CHECKPOINT_DIR = OUTPUTS_DIR / "checkpoints_dgcnn_comparison"
METRICS_DIR = OUTPUTS_DIR / "metrics_dgcnn_comparison"
assert CHECKPOINT_DIR.exists(), f"{CHECKPOINT_DIR} not found -- run 07i first."
EXPLAIN_DIR = OUTPUTS_DIR / "explain_dgcnn_comparison"
EXPLAIN_DIR.mkdir(parents=True, exist_ok=True)
# 08g's dir, read-only here -- only used later by the comparison cell.
CAPACITY_REVISION_EXPLAIN_DIR = OUTPUTS_DIR / "explain_capacity_revision"

device = "cuda" if torch.cuda.is_available() else "cpu"
HEAD_DEPTH = model_cfg.get("head_depth", "mlp2")
CONV_TYPE = model_cfg.get("conv_type", "gatv2")
READOUT = model_cfg.get("readout", "dgcnn")

N_PER_CATEGORY = 2
GNNEXPLAINER_EPOCHS = 100
GNNEXPLAINER_LR = 0.05
EXPLAIN_SEED = 42

print("Device:", device, "| head_depth:", HEAD_DEPTH, "| readout:", READOUT)
print("Reading checkpoints from:", CHECKPOINT_DIR)
print("Writing explanations to:", EXPLAIN_DIR)
print(f"N_PER_CATEGORY={N_PER_CATEGORY}, gnnexplainer_epochs={GNNEXPLAINER_EPOCHS}")

In [ ]:
import json
import copy
import random
import pandas as pd
import graph_datasets as ds
import models
import explain
import unified_graph as ug

SVG_DIR = COMBINED_PROCESSED_DIR / "svg_graphs"
TVG_DIR = COMBINED_PROCESSED_DIR / "tvg_graphs"
INDEX_PATH = COMBINED_PROCESSED_DIR / "dataset_index.parquet"
index_df = pd.read_parquet(INDEX_PATH)
assert "city" in index_df.columns, (
    f"'{INDEX_PATH}' has no 'city' column -- this notebook needs 05's combined, "
    "multi-city dataset_index.parquet, not a single-city index.")
print(f"Dataset: {len(index_df)} points available for lookup")

_ref_cache_dir = INTERIM_DIR / "osm_cache" / CITIES[0]
with open(_ref_cache_dir / "highway_vocab.json") as f:
    HIGHWAY_VOCAB_SIZE = len(json.load(f))
with open(_ref_cache_dir / "building_type_vocab.json") as f:
    BUILDING_TYPE_VOCAB_SIZE = len(json.load(f))
print(f"Unified vocab (post-04b): highway={HIGHWAY_VOCAB_SIZE}, building_type={BUILDING_TYPE_VOCAB_SIZE}")

## Reconstruct `k` -- copied verbatim from 07i's own diagnostic cell

Must reproduce the exact `k` values `07i` actually trained with, or the
Conv1d/Linear layers inside `DGCNNReadout` won't match
`best_model.pt`'s saved state_dict shapes. Same sample (`n=500,
random_state=42`), same rule (`k = max(floor, p40)`) -- deterministic
given the same underlying dataset.

In [ ]:
import numpy as np
from graph_datasets import DualGraphDataset

_lookup_dataset = DualGraphDataset(index_df, SVG_DIR, TVG_DIR)

def _total_node_count(hetero_data, node_types):
    return sum(int(hetero_data[nt].x.shape[0]) for nt in node_types if nt in hetero_data.node_types)

SAMPLE_N = min(500, len(_lookup_dataset))
sample_idx = index_df.sample(n=SAMPLE_N, random_state=42).index

_tvg_node_types_no_peer = [nt for nt in models.TVG_NODE_TYPES if nt != "peer_incident"]

svg_counts, tvg_counts, unified_counts = [], [], []
for i in sample_idx:
    svg_d, tvg_d, _label, _pid = _lookup_dataset[i]
    svg_counts.append(_total_node_count(svg_d, models.SVG_NODE_TYPES))
    tvg_counts.append(_total_node_count(tvg_d, _tvg_node_types_no_peer))
    merged_d = ug.merge_svg_tvg(svg_d, tvg_d)
    unified_counts.append(_total_node_count(merged_d, models.UNIFIED_NODE_TYPES))

def _compute_k(counts, conv2_kernel, label):
    counts = np.asarray(counts)
    p40 = np.percentile(counts, 40)
    floor_k = (conv2_kernel - 1) * 2 + 2
    k = max(floor_k, int(np.ceil(p40)))
    print(f"{label:8s} | p40={p40:5.1f} | conv2_kernel={conv2_kernel} floor={floor_k:2d} -> k={k:3d}")
    return k

SVG_DGCNN_K = _compute_k(svg_counts, model_cfg.get("svg_dgcnn_conv2_kernel", 5), "SVG")
TVG_DGCNN_K = _compute_k(tvg_counts, model_cfg.get("tvg_dgcnn_conv2_kernel", 3), "TVG")
UNIFIED_DGCNN_K = _compute_k(unified_counts, model_cfg.get("unified_dgcnn_conv2_kernel", 5), "Unified")

In [ ]:
svg_kwargs = dict(hidden_dim=model_cfg.get("hidden_dim", 128), heads=model_cfg.get("heads", 4),
                   num_layers=model_cfg.get("svg_layers", 2), dropout=model_cfg.get("dropout", 0.3),
                   signage_vocab=5, light_pole_vocab=4, road_marking_vocab=2,
                   cat_embed_dim=model_cfg.get("cat_embed_dim", 4), conv_type=CONV_TYPE,
                   readout=READOUT, dgcnn_k=SVG_DGCNN_K,
                   dgcnn_conv2_kernel=model_cfg.get("svg_dgcnn_conv2_kernel", 5))
tvg_kwargs = dict(hidden_dim=model_cfg.get("hidden_dim", 128), heads=model_cfg.get("heads", 4),
                   num_layers=model_cfg.get("tvg_layers", 2), dropout=model_cfg.get("dropout", 0.3),
                   building_type_vocab=BUILDING_TYPE_VOCAB_SIZE, highway_vocab=HIGHWAY_VOCAB_SIZE,
                   building_type_embed_dim=model_cfg.get("building_type_embed_dim", 16),
                   highway_embed_dim=model_cfg.get("highway_embed_dim", 8), conv_type=CONV_TYPE,
                   readout=READOUT, dgcnn_k=TVG_DGCNN_K,
                   dgcnn_conv2_kernel=model_cfg.get("tvg_dgcnn_conv2_kernel", 3))
FUSION_DIM = model_cfg.get("fusion_dim", 256)
HEAD_HIDDEN = model_cfg.get("head_hidden", 256)
HEAD_DROPOUT = model_cfg.get("head_dropout", 0.3)
print("svg_kwargs:", svg_kwargs)
print("tvg_kwargs:", tvg_kwargs)
print("unified_dgcnn_k:", UNIFIED_DGCNN_K)

## Helpers: load a trained scenario's model+stats, pick which points to explain

Same logic as `08g`'s helpers, with one addition: scenario F needs
`unified_dgcnn_k` passed explicitly to `build_model()` (mirrors 07i's own
call -- see `build_model`'s `unified_dgcnn_k` override, without which
`svg_kwargs`'s `dgcnn_k` would silently win the kwarg merge).

In [ ]:
def load_scenario_model(scenario, use_ablation=False):
    tag = f"{scenario}_{HEAD_DEPTH}" + ("_ablation" if use_ablation else "")
    model_path = CHECKPOINT_DIR / f"{tag}_best_model.pt"
    stats_path = CHECKPOINT_DIR / f"{tag}_best_model_stats.pt"
    meta_path = CHECKPOINT_DIR / f"{tag}_best_model_meta.json"
    assert model_path.exists(), f"{model_path} not found -- run 07i's scenario {scenario} cell first."
    assert stats_path.exists(), (
        f"{stats_path} not found -- this checkpoint predates the best_model_stats.pt "
        f"persistence change; rerun 07i's scenario {scenario} cell with the current train.py.")

    model = models.build_model(scenario, fusion_dim=FUSION_DIM, head_depth=HEAD_DEPTH,
                                head_hidden=HEAD_HIDDEN, head_dropout=HEAD_DROPOUT,
                                use_ablation=use_ablation, svg_kwargs=svg_kwargs, tvg_kwargs=tvg_kwargs,
                                unified_dgcnn_k=UNIFIED_DGCNN_K)
    model.load_state_dict(torch.load(model_path, map_location="cpu", weights_only=False))
    model.eval()
    stats = torch.load(stats_path, map_location="cpu", weights_only=False)
    meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}
    best_repeat = meta.get("repeat")
    return model, stats, best_repeat, tag


def pick_points_to_explain(tag, best_repeat, n_per_category=N_PER_CATEGORY, seed=EXPLAIN_SEED):
    if best_repeat is None:
        print(f"  [{tag}] no best_model_meta.json repeat recorded -- skipping point selection.")
        return []
    pred_path = CHECKPOINT_DIR / f"{tag}_history" / f"repeat{best_repeat}_test_predictions.json"
    if not pred_path.exists():
        print(f"  [{tag}] {pred_path} not found -- skipping.")
        return []
    records = json.loads(pred_path.read_text())
    rng = random.Random(seed)
    by_category = {}
    for r in records:
        by_category.setdefault(r["category"], []).append(r)
    picked = []
    for cat, rows in sorted(by_category.items()):
        sample = rng.sample(rows, min(n_per_category, len(rows)))
        picked.extend({"point_id": r["point_id"], "category": cat} for r in sample)
    return picked

## Run both explanation techniques across scenarios A-F

In [ ]:
all_records = []

for scenario in ["A", "B", "C", "D", "E", "F"]:
    print(f"\n=== Scenario {scenario} ===")
    model, stats, best_repeat, tag = load_scenario_model(scenario)
    print(f"  loaded {tag} (best repeat={best_repeat})")

    points = pick_points_to_explain(tag, best_repeat)
    print(f"  explaining {len(points)} points: "
          f"{ {c: sum(1 for p in points if p['category']==c) for c in sorted(set(p['category'] for p in points))} }")

    for p in points:
        pid, cat = p["point_id"], p["category"]
        svg_raw = torch.load(SVG_DIR / f"{pid}.pt", weights_only=False)
        tvg_raw = torch.load(TVG_DIR / f"{pid}.pt", weights_only=False)
        svg_norm, tvg_norm = ds.apply_normalization(copy.deepcopy(svg_raw), copy.deepcopy(tvg_raw), stats)

        try:
            records = explain.explain_scenario_point(
                model, scenario, svg_norm, tvg_norm, point_id=pid, category=cat,
                gnnexplainer_epochs=GNNEXPLAINER_EPOCHS, gnnexplainer_lr=GNNEXPLAINER_LR)
            all_records.extend(records)
        except Exception as e:
            print(f"    !! failed to explain {pid} ({cat}): {type(e).__name__}: {e}")

print(f"\nTotal explained (point, branch) records: {len(all_records)}")

## Aggregate into one tidy CSV

In [ ]:
explain_df = explain.aggregate_explanations(all_records)
explain_df.to_csv(EXPLAIN_DIR / "explanations_dgcnn_comparison.csv", index=False)
print(f"Saved {len(explain_df)} rows to {EXPLAIN_DIR / 'explanations_dgcnn_comparison.csv'}")
display(explain_df.head(10))

## The question this notebook exists to answer: does DGCNN actually drop the anchor?

Unlike `08g`, `DGCNNReadout` has NO guarantee the anchor node
(`ego`/`incident`) survives SortPooling's top-`k` cutoff. This cell
computes the same anchor-importance summary `08g` does, so the two can
be compared directly -- a lower anchor importance here (especially on
FP/FN points) would be direct empirical support for the "no anchor
guarantee" concern; a similar or higher anchor importance would suggest
the model still learns to rank the anchor highly on its own, even
without the architectural guarantee.

In [ ]:
anchor_types = {"A": "ego", "B": "incident", "C_svg": "ego", "C_tvg": "incident",
                 "D_svg": "ego", "D_tvg": "incident", "E_svg": "ego", "E_tvg": "incident",
                 "F": "incident"}

gnne_node = explain_df[(explain_df["source"] == "gnnexplainer") & (explain_df["kind"] == "node")]
rows = []
for scen, anchor_nt in anchor_types.items():
    sub = gnne_node[gnne_node["scenario"] == scen]
    anchor_rows = sub[sub["type"] == anchor_nt]
    other_rows = sub[sub["type"] != anchor_nt]
    if len(anchor_rows) == 0:
        continue
    rows.append({
        "scenario": scen, "anchor_type": anchor_nt,
        "anchor_mean_importance": anchor_rows["mean_value"].mean(),
        "other_types_mean_importance": other_rows["mean_value"].mean() if len(other_rows) else float("nan"),
        "n_points": anchor_rows["point_id"].nunique(),
    })
    # split by category too -- does the anchor's importance drop specifically on wrong predictions?
    for cat in ["TP", "TN", "FP", "FN"]:
        cat_anchor = anchor_rows[anchor_rows["category"] == cat]
        if len(cat_anchor):
            rows[-1][f"anchor_importance_{cat}"] = cat_anchor["mean_value"].mean()

anchor_summary_df = pd.DataFrame(rows)
anchor_summary_df.to_csv(EXPLAIN_DIR / "anchor_importance_summary.csv", index=False)
display(anchor_summary_df)

capacity_revision_anchor_path = CAPACITY_REVISION_EXPLAIN_DIR / "anchor_importance_summary.csv"
if capacity_revision_anchor_path.exists():
    gatv2_anchor_df = pd.read_csv(capacity_revision_anchor_path)
    compare = anchor_summary_df[["scenario", "anchor_type", "anchor_mean_importance"]].merge(
        gatv2_anchor_df[["scenario", "anchor_mean_importance"]],
        on="scenario", suffixes=("_dgcnn", "_pool_anchor"))
    compare["anchor_importance_delta"] = (
        compare["anchor_mean_importance_dgcnn"] - compare["anchor_mean_importance_pool_anchor"])
    compare.to_csv(EXPLAIN_DIR / "anchor_importance_dgcnn_vs_pool_anchor.csv", index=False)
    display(compare)
    n_lower = (compare["anchor_importance_delta"] < 0).sum()
    print(f"DGCNN's anchor importance is LOWER than pool_anchor's on {n_lower}/{len(compare)} scenarios.")
else:
    print("08g's anchor_importance_summary.csv not found yet -- run 08g first to compare.")

In [ ]:
print("08i explainability run complete.")
print(f"Explained {explain_df['point_id'].nunique()} unique points across "
      f"{explain_df['scenario'].nunique()} scenario/branch tags.")
print(f"k values used (reconstructed to match 07i): svg={SVG_DGCNN_K} tvg={TVG_DGCNN_K} unified={UNIFIED_DGCNN_K}")
print(f"See {EXPLAIN_DIR / 'explanations_dgcnn_comparison.csv'} for the full tidy table,")
print(f"and {EXPLAIN_DIR / 'anchor_importance_dgcnn_vs_pool_anchor.csv'} for the direct")
print("anchor-node-importance comparison against 08g.")